In [ ]:
# =============================================================================
# 🧹 NETTOYAGE CACHE & MÉMOIRE (exécuter en premier sur SSH!)
# =============================================================================
import os
import gc
import shutil
import glob

def cleanup_cache(verbose=True):
    """Nettoie les fichiers cache et libère la mémoire."""
    cleaned_size = 0
    
    # 1. Nettoyer __pycache__
    for pycache in glob.glob('**/__pycache__', recursive=True):
        try:
            shutil.rmtree(pycache)
            if verbose:
                print(f"🗑️ Supprimé: {pycache}")
        except:
            pass
    
    # 2. Nettoyer fichiers .pyc
    for pyc in glob.glob('**/*.pyc', recursive=True):
        try:
            os.remove(pyc)
        except:
            pass
    
    # 3. Nettoyer cache HuggingFace (GROS!)
    hf_cache = os.path.expanduser('~/.cache/huggingface')
    if os.path.exists(hf_cache):
        size = sum(os.path.getsize(os.path.join(dp, f)) 
                   for dp, dn, fn in os.walk(hf_cache) for f in fn)
        cleaned_size += size
        if verbose:
            print(f"📦 Cache HuggingFace: {size / 1e9:.2f} GB")
        # Décommenter pour supprimer (attention!)
        # shutil.rmtree(hf_cache)
    
    # 4. Nettoyer cache PyTorch
    torch_cache = os.path.expanduser('~/.cache/torch')
    if os.path.exists(torch_cache):
        size = sum(os.path.getsize(os.path.join(dp, f)) 
                   for dp, dn, fn in os.walk(torch_cache) for f in fn)
        if verbose:
            print(f"🔥 Cache PyTorch: {size / 1e6:.2f} MB")
    
    # 5. Garbage collection Python
    gc.collect()
    
    # 6. Nettoyer cache CUDA si disponible
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            if verbose:
                print(f"🖥️ CUDA cache vidé")
    except:
        pass
    
    # 7. Afficher espace disque
    total, used, free = shutil.disk_usage('/')
    if verbose:
        print(f"\n💾 Espace disque: {free / 1e9:.1f} GB libre / {total / 1e9:.1f} GB total")
    
    return free / 1e9

# Exécuter le nettoyage
free_gb = cleanup_cache()
print(f"\n✅ Nettoyage terminé!")

In [ ]:
# =============================================================================
# 📦 IMPORTS
# =============================================================================
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# Optuna
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    print("⚠️ Optuna non installé. Run: pip install optuna")
    OPTUNA_AVAILABLE = False

# Configuration
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"🔢 PyTorch: {torch.__version__}")

## 📥 1. Chargement des Données

In [ ]:
# =============================================================================
# 📂 CONFIGURATION DES CHEMINS
# =============================================================================
DATA_DIR = './data'
EMB_DIR = os.path.join(DATA_DIR, 'embeddings')
SUBMISSION_DIR = './submission'

os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Fichiers de données
paths = {
    'train_features': os.path.join(DATA_DIR, 'train_features.csv'),
    'test_features': os.path.join(DATA_DIR, 'test_features.csv'),
    'y_train': os.path.join(DATA_DIR, 'y_train.npy'),
    'train_emb': os.path.join(EMB_DIR, 'X_train_text_embeddings.npy'),
    'test_emb': os.path.join(EMB_DIR, 'X_kaggle_text_embeddings.npy'),
    'train_emb_ml': os.path.join(EMB_DIR, 'X_train_multilayer_embeddings.npy'),
    'test_emb_ml': os.path.join(EMB_DIR, 'X_kaggle_multilayer_embeddings.npy'),
}

# Vérifier fichiers
print("📁 Vérification des fichiers:")
for name, path in paths.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"  {exists} {name}")

In [ ]:
# =============================================================================
# 📊 TOP FEATURES (basé sur feature_engineering.ipynb - LightGBM importance)
# =============================================================================

TOP_FEATURES = [
    # ⭐ TOP 6 features (importance > 500 dans LightGBM)
    'user_description_length',   # importance=736
    'tweets_per_favourites',     # importance=699
    'user_favourites_count',     # importance=646
    'user_statuses_count',       # importance=639
    'user_listed_count',         # importance=607
    'listed_per_status',         # importance=553
    
    # Log transforms (haute corrélation avec label)
    'log_user_listed',           # corr=0.606
    'log_user_statuses',         # corr=0.439
    'log_user_favourites',
    
    # Engagement metrics
    'total_engagement', 'log_total_engagement',
    'retweet_count', 'favorite_count', 'reply_count', 'quote_count',
    'log_retweet_count', 'log_favorite_count',
    
    # ⭐ Binary user features (très discriminantes)
    'user_has_url',              # Observer=16%, Influencer=56%
    'user_has_banner',           # Observer=73%, Influencer=92%
    'user_has_location',         # Observer=58%, Influencer=75%
    'user_has_long_desc',
    'user_default_profile', 'user_default_profile_image',
    
    # Source device (discriminant)
    'is_iphone', 'is_android', 'is_web', 'is_tweetdeck', 'is_bot_source',
    
    # Text features
    'tweet_length', 'word_count', 'uppercase_ratio',
    'hashtag_count', 'is_hashtag_heavy',
    'mention_count', 'is_mention_heavy',
    'emoji_count', 'is_emoji_heavy',
    'exclamation_count', 'question_count', 'has_multiple_exclamations',
    'url_count', 'has_url',
    
    # Content detection
    'is_reply', 'is_in_reply', 'is_reply_to_someone',
    'has_rt_qt', 'is_quote_status', 'has_quoted_status',
    'has_call_to_action', 'has_self_promotion', 'has_media_reference',
    
    # Entities
    'entities_hashtags', 'entities_urls', 'entities_mentions', 'entities_symbols',
    
    # Other
    'first_person_count', 'is_long_tweet', 'is_short_tweet'
]

print(f"📊 {len(TOP_FEATURES)} features définies")

In [ ]:
# =============================================================================
# 📥 CHARGER LES DONNÉES
# =============================================================================

def clean_array(arr):
    """Nettoie NaN/inf et convertit en float32."""
    arr = np.asarray(arr)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
    return arr.astype(np.float32)

# 1. Charger features structurées
print("📊 Chargement des features...")
train_df = pd.read_csv(paths['train_features'])
test_df = pd.read_csv(paths['test_features'])

# Filtrer features disponibles
available_features = [f for f in TOP_FEATURES if f in train_df.columns]
print(f"   Features disponibles: {len(available_features)}/{len(TOP_FEATURES)}")

X_train_feat = clean_array(train_df[available_features].fillna(0).values)
X_test_feat = clean_array(test_df[available_features].fillna(0).values)

# 2. Charger embeddings (si disponibles)
USE_EMBEDDINGS = os.path.exists(paths['train_emb'])
USE_MULTILAYER = os.path.exists(paths['train_emb_ml'])

if USE_MULTILAYER:
    print("📊 Chargement embeddings multi-layer...")
    X_train_emb = clean_array(np.load(paths['train_emb_ml']))
    X_test_emb = clean_array(np.load(paths['test_emb_ml']))
elif USE_EMBEDDINGS:
    print("📊 Chargement embeddings standard...")
    X_train_emb = clean_array(np.load(paths['train_emb']))
    X_test_emb = clean_array(np.load(paths['test_emb']))
else:
    print("⚠️ Pas d'embeddings - utilisation features uniquement")
    X_train_emb = None
    X_test_emb = None

# 3. Charger labels
y_full = np.load(paths['y_train'])

# 4. Normaliser features
scaler = StandardScaler()
X_train_feat_scaled = scaler.fit_transform(X_train_feat)
X_test_feat_scaled = scaler.transform(X_test_feat)

# 5. Combiner si embeddings disponibles
if X_train_emb is not None:
    X_train_combined = np.hstack([X_train_feat_scaled, X_train_emb])
    X_test_combined = np.hstack([X_test_feat_scaled, X_test_emb])
else:
    X_train_combined = X_train_feat_scaled
    X_test_combined = X_test_feat_scaled

print(f"\n📊 Dimensions finales:")
print(f"   X_train (features only): {X_train_feat_scaled.shape}")
print(f"   X_train (combined): {X_train_combined.shape}")
print(f"   y_train: {y_full.shape}, Classes: {np.bincount(y_full)}")

In [ ]:
# =============================================================================
# 📊 TRAIN/VAL SPLIT
# =============================================================================

X_train, X_val, y_train, y_val = train_test_split(
    X_train_feat_scaled, y_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_full
)

# Pour le NN (avec embeddings si disponibles)
X_train_nn, X_val_nn, _, _ = train_test_split(
    X_train_combined, y_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_full
)

print(f"📊 Split:")
print(f"   Train: {X_train.shape[0]} samples")
print(f"   Val: {X_val.shape[0]} samples")

## 🧠 2. Définition des Modèles

In [ ]:
# =============================================================================
# 🌲 MODÈLE 1: LightGBM
# =============================================================================

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

print("✅ LightGBM configuré")

In [ ]:
# =============================================================================
# 🌲 MODÈLE 2: XGBoost
# =============================================================================

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    eval_metric='logloss',
    use_label_encoder=False
)

print("✅ XGBoost configuré")

In [ ]:
# =============================================================================
# 📈 MODÈLE 3: Logistic Regression
# =============================================================================

lr_model = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1
)

print("✅ Logistic Regression configuré")

In [ ]:
# =============================================================================
# 🧠 MODÈLE 4: Neural Network (PyTorch)
# =============================================================================

class TweetMLP(nn.Module):
    """MLP pour classification de tweets."""
    def __init__(self, input_dim, hidden_dims=[256, 128, 64], dropout=0.3):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)


class PyTorchClassifier(BaseEstimator, ClassifierMixin):
    """Wrapper sklearn pour modèle PyTorch."""
    
    def __init__(self, input_dim, epochs=30, batch_size=128, lr=0.001):
        self.input_dim = input_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.model = None
        self.device = device
    
    def fit(self, X, y):
        # Créer modèle
        self.model = TweetMLP(self.input_dim).to(self.device)
        
        # Data
        X_t = torch.FloatTensor(X).to(self.device)
        y_t = torch.LongTensor(y).to(self.device)
        dataset = TensorDataset(X_t, y_t)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        # Training
        criterion = nn.CrossEntropyLoss()
        optimizer = AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        scheduler = OneCycleLR(optimizer, max_lr=self.lr*10, 
                               steps_per_epoch=len(loader), epochs=self.epochs)
        
        self.model.train()
        for epoch in range(self.epochs):
            for X_batch, y_batch in loader:
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
        
        return self
    
    def predict(self, X):
        self.model.eval()
        X_t = torch.FloatTensor(X).to(self.device)
        with torch.no_grad():
            outputs = self.model(X_t)
            preds = torch.argmax(outputs, dim=1)
        return preds.cpu().numpy()
    
    def predict_proba(self, X):
        self.model.eval()
        X_t = torch.FloatTensor(X).to(self.device)
        with torch.no_grad():
            outputs = self.model(X_t)
            probs = torch.softmax(outputs, dim=1)
        return probs.cpu().numpy()


nn_model = PyTorchClassifier(
    input_dim=X_train_combined.shape[1],
    epochs=30,
    batch_size=128,
    lr=0.001
)

print(f"✅ Neural Network configuré (input_dim={X_train_combined.shape[1]})")

## 📊 3. Évaluation Individuelle des Modèles

In [ ]:
# =============================================================================
# 📊 CROSS-VALIDATION INDIVIDUELLE
# =============================================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}

print("📊 Évaluation Cross-Validation (5-fold):")
print("=" * 50)

# LightGBM
print("\n🌲 LightGBM...")
scores_lgbm = cross_val_score(lgbm_model, X_train_feat_scaled, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
results['LightGBM'] = scores_lgbm
print(f"   Accuracy: {scores_lgbm.mean():.4f} (+/- {scores_lgbm.std()*2:.4f})")

# XGBoost
print("\n🌲 XGBoost...")
scores_xgb = cross_val_score(xgb_model, X_train_feat_scaled, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
results['XGBoost'] = scores_xgb
print(f"   Accuracy: {scores_xgb.mean():.4f} (+/- {scores_xgb.std()*2:.4f})")

# Logistic Regression
print("\n📈 Logistic Regression...")
scores_lr = cross_val_score(lr_model, X_train_feat_scaled, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
results['LogReg'] = scores_lr
print(f"   Accuracy: {scores_lr.mean():.4f} (+/- {scores_lr.std()*2:.4f})")

print("\n" + "=" * 50)
print("📊 Résumé:")
for name, scores in results.items():
    print(f"   {name}: {scores.mean():.4f}")

In [ ]:
# =============================================================================
# 🧠 ÉVALUATION NEURAL NETWORK (sur combined features)
# =============================================================================

print("\n🧠 Neural Network (train/val split)...")

# Entraîner
nn_model_eval = PyTorchClassifier(
    input_dim=X_train_combined.shape[1],
    epochs=30,
    batch_size=128,
    lr=0.001
)
nn_model_eval.fit(X_train_nn, y_train)

# Évaluer
y_val_pred_nn = nn_model_eval.predict(X_val_nn)
nn_acc = accuracy_score(y_val, y_val_pred_nn)
print(f"   Val Accuracy: {nn_acc:.4f}")

# Libérer mémoire
del nn_model_eval
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 🗳️ 4. Ensemble Methods

In [ ]:
# =============================================================================
# 🗳️ VOTING CLASSIFIER (Soft Voting)
# =============================================================================

print("🗳️ VotingClassifier (Soft Voting)...")

voting_clf = VotingClassifier(
    estimators=[
        ('lgbm', lgbm_model),
        ('xgb', xgb_model),
        ('lr', lr_model)
    ],
    voting='soft',
    n_jobs=-1
)

# Cross-validation
scores_voting = cross_val_score(voting_clf, X_train_feat_scaled, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"   CV Accuracy: {scores_voting.mean():.4f} (+/- {scores_voting.std()*2:.4f})")

In [ ]:
# =============================================================================
# 📚 STACKING CLASSIFIER
# =============================================================================

print("📚 StackingClassifier...")

stacking_clf = StackingClassifier(
    estimators=[
        ('lgbm', LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, 
                                random_state=SEED, n_jobs=-1, verbose=-1)),
        ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=7,
                              random_state=SEED, n_jobs=-1, eval_metric='logloss',
                              use_label_encoder=False)),
        ('lr', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

# Cross-validation
scores_stacking = cross_val_score(stacking_clf, X_train_feat_scaled, y_full, 
                                   cv=cv, scoring='accuracy', n_jobs=-1)
print(f"   CV Accuracy: {scores_stacking.mean():.4f} (+/- {scores_stacking.std()*2:.4f})")

## 🎯 5. Optimisation Optuna des Poids

In [ ]:
# =============================================================================
# 🎯 OPTUNA - OPTIMISATION DES POIDS D'ENSEMBLE
# =============================================================================

if OPTUNA_AVAILABLE:
    print("🎯 Optimisation Optuna des poids d'ensemble...")
    
    # Entraîner les modèles de base
    print("   Training base models...")
    lgbm_model.fit(X_train, y_train)
    xgb_model.fit(X_train, y_train)
    lr_model.fit(X_train, y_train)
    
    # Obtenir probabilités sur validation
    lgbm_proba = lgbm_model.predict_proba(X_val)[:, 1]
    xgb_proba = xgb_model.predict_proba(X_val)[:, 1]
    lr_proba = lr_model.predict_proba(X_val)[:, 1]
    
    def objective(trial):
        """Objective function pour Optuna."""
        # Suggérer poids
        w_lgbm = trial.suggest_float('w_lgbm', 0.1, 0.8)
        w_xgb = trial.suggest_float('w_xgb', 0.1, 0.8)
        w_lr = trial.suggest_float('w_lr', 0.0, 0.4)
        
        # Normaliser
        total = w_lgbm + w_xgb + w_lr
        w_lgbm, w_xgb, w_lr = w_lgbm/total, w_xgb/total, w_lr/total
        
        # Weighted average
        ensemble_proba = w_lgbm * lgbm_proba + w_xgb * xgb_proba + w_lr * lr_proba
        ensemble_pred = (ensemble_proba > 0.5).astype(int)
        
        return accuracy_score(y_val, ensemble_pred)
    
    # Optimiser
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100, show_progress_bar=True)
    
    # Meilleurs poids
    best_params = study.best_params
    total = best_params['w_lgbm'] + best_params['w_xgb'] + best_params['w_lr']
    
    BEST_WEIGHTS = {
        'lgbm': best_params['w_lgbm'] / total,
        'xgb': best_params['w_xgb'] / total,
        'lr': best_params['w_lr'] / total
    }
    
    print(f"\n🎯 Meilleurs poids:")
    for name, weight in BEST_WEIGHTS.items():
        print(f"   {name}: {weight:.3f}")
    print(f"\n🎯 Meilleure accuracy: {study.best_value:.4f}")
    
else:
    print("⚠️ Optuna non disponible - utilisation poids égaux")
    BEST_WEIGHTS = {'lgbm': 0.4, 'xgb': 0.4, 'lr': 0.2}

## 📊 6. Résumé des Résultats

In [ ]:
# =============================================================================
# 📊 RÉSUMÉ DES RÉSULTATS
# =============================================================================

print("\n" + "=" * 60)
print("📊 RÉSUMÉ DES RÉSULTATS")
print("=" * 60)

summary = pd.DataFrame({
    'Modèle': ['LightGBM', 'XGBoost', 'LogReg', 'Voting', 'Stacking'],
    'CV Accuracy': [
        results['LightGBM'].mean(),
        results['XGBoost'].mean(),
        results['LogReg'].mean(),
        scores_voting.mean(),
        scores_stacking.mean()
    ],
    'Std': [
        results['LightGBM'].std(),
        results['XGBoost'].std(),
        results['LogReg'].std(),
        scores_voting.std(),
        scores_stacking.std()
    ]
}).sort_values('CV Accuracy', ascending=False)

print(summary.to_string(index=False))

# Meilleur modèle
best_model_name = summary.iloc[0]['Modèle']
best_accuracy = summary.iloc[0]['CV Accuracy']
print(f"\n🏆 Meilleur modèle: {best_model_name} ({best_accuracy:.4f})")

## 🚀 7. Génération des Submissions

In [ ]:
# =============================================================================
# 🚀 ENTRAÎNEMENT FINAL & PRÉDICTIONS
# =============================================================================

print("🚀 Entraînement final sur toutes les données...")

# Ré-entraîner sur toutes les données
lgbm_final = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_final.fit(X_train_feat_scaled, y_full)
print("   ✅ LightGBM")

xgb_final = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, eval_metric='logloss', use_label_encoder=False
)
xgb_final.fit(X_train_feat_scaled, y_full)
print("   ✅ XGBoost")

lr_final = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, n_jobs=-1)
lr_final.fit(X_train_feat_scaled, y_full)
print("   ✅ LogReg")

In [ ]:
# =============================================================================
# 📤 GÉNÉRATION SUBMISSION - WEIGHTED ENSEMBLE
# =============================================================================

print("\n📤 Génération submission weighted ensemble...")

# Prédictions probas sur test
lgbm_test_proba = lgbm_final.predict_proba(X_test_feat_scaled)[:, 1]
xgb_test_proba = xgb_final.predict_proba(X_test_feat_scaled)[:, 1]
lr_test_proba = lr_final.predict_proba(X_test_feat_scaled)[:, 1]

# Weighted ensemble
ensemble_proba = (
    BEST_WEIGHTS['lgbm'] * lgbm_test_proba + 
    BEST_WEIGHTS['xgb'] * xgb_test_proba + 
    BEST_WEIGHTS['lr'] * lr_test_proba
)
ensemble_pred = (ensemble_proba > 0.5).astype(int)

# Créer submission
submission = pd.DataFrame({
    'ID': test_df['challenge_id'].astype(int),
    'Prediction': ensemble_pred
})

# Sauvegarder
submission.to_csv(os.path.join(SUBMISSION_DIR, 'ensemble_weighted.csv'), index=False)
print(f"✅ Sauvegardé: submission/ensemble_weighted.csv")
print(f"   Distribution: {np.bincount(ensemble_pred)}")

In [ ]:
# =============================================================================
# 📤 GÉNÉRATION SUBMISSION - STACKING
# =============================================================================

print("\n📤 Génération submission stacking...")

# Entraîner stacking sur toutes les données
stacking_final = StackingClassifier(
    estimators=[
        ('lgbm', LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, 
                                random_state=SEED, n_jobs=-1, verbose=-1)),
        ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=7,
                              random_state=SEED, n_jobs=-1, eval_metric='logloss',
                              use_label_encoder=False)),
        ('lr', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    cv=5,
    n_jobs=-1
)

stacking_final.fit(X_train_feat_scaled, y_full)
stacking_pred = stacking_final.predict(X_test_feat_scaled)

# Créer submission
submission_stacking = pd.DataFrame({
    'ID': test_df['challenge_id'].astype(int),
    'Prediction': stacking_pred
})

submission_stacking.to_csv(os.path.join(SUBMISSION_DIR, 'ensemble_stacking.csv'), index=False)
print(f"✅ Sauvegardé: submission/ensemble_stacking.csv")
print(f"   Distribution: {np.bincount(stacking_pred)}")

In [ ]:
# =============================================================================
# 📤 GÉNÉRATION SUBMISSION - MEILLEUR SINGLE MODEL (LightGBM)
# =============================================================================

print("\n📤 Génération submission LightGBM seul...")

lgbm_pred = lgbm_final.predict(X_test_feat_scaled)

submission_lgbm = pd.DataFrame({
    'ID': test_df['challenge_id'].astype(int),
    'Prediction': lgbm_pred
})

submission_lgbm.to_csv(os.path.join(SUBMISSION_DIR, 'submission_lgbm.csv'), index=False)
print(f"✅ Sauvegardé: submission/submission_lgbm.csv")
print(f"   Distribution: {np.bincount(lgbm_pred)}")

In [ ]:
# =============================================================================
# 🧹 NETTOYAGE FINAL
# =============================================================================

print("\n🧹 Nettoyage mémoire...")

# Libérer mémoire
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Afficher espace libre
total, used, free = shutil.disk_usage('/')
print(f"💾 Espace disque: {free / 1e9:.1f} GB libre")

print("\n" + "=" * 60)
print("✅ TERMINÉ!")
print("=" * 60)
print("\n📁 Fichiers générés:")
for f in sorted(os.listdir(SUBMISSION_DIR)):
    if f.endswith('.csv'):
        path = os.path.join(SUBMISSION_DIR, f)
        size = os.path.getsize(path) / 1e6
        print(f"   📄 {f} ({size:.2f} MB)")

## 📋 Résumé

### Modèles utilisés:
- **LightGBM**: Gradient Boosting rapide, très efficace sur features tabulaires
- **XGBoost**: Gradient Boosting robuste avec régularisation
- **Logistic Regression**: Baseline linéaire stable
- **Neural Network**: MLP PyTorch pour embeddings

### Méthodes d'ensemble:
- **VotingClassifier**: Moyenne des probabilités (soft voting)
- **StackingClassifier**: Meta-learner sur prédictions des modèles de base
- **Weighted Ensemble**: Poids optimisés par Optuna

### Fichiers de soumission:
- `submission/ensemble_weighted.csv` - Ensemble pondéré (Optuna)
- `submission/ensemble_stacking.csv` - Stacking avec meta-learner
- `submission/submission_lgbm.csv` - LightGBM seul

---
*Dernière mise à jour: 9 décembre 2025*